# Notebook 17 – Complete ML Workflow

**Dataset:** Online Retail transactions (`data.csv`, same as Notebook 16)

**Task:** Predict whether an order is from the **United Kingdom** or not, using order details (`Quantity`, `UnitPrice`).


## 1. Define the Problem

This is a **binary classification** problem: given order details, predict `is_UK` (1 = order from United Kingdom, 0 = order from another country). Knowing this helps a retailer understand shipping/region patterns and target other countries.

In [1]:
print("Goal: Binary classification - is the order from the UK?")

Goal: Binary classification - is the order from the UK?


## 2. Load Dataset

We load the raw CSV file into a pandas DataFrame.

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv('data.csv', encoding='latin1')
df.shape

(541909, 8)

## 3. Understand Dataset

Look at column types, sample rows, and basic info to understand what we're working with.

In [3]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 67.3 MB


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## 4. Perform EDA (Exploratory Data Analysis)

Check the target distribution and basic statistics of the numeric features.

In [4]:
print(df['Country'].value_counts().head())
df[['Quantity', 'UnitPrice']].describe()

Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Name: count, dtype: int64


,Quantity,UnitPrice
count,541909.000000,541909.000000
mean,9.552250,4.611114
std,218.081158,96.759853
min,-80995.000000,-11062.060000
25%,1.000000,1.250000
50%,3.000000,2.080000
75%,10.000000,4.130000
max,80995.000000,38970.000000


## 5. Clean Data

Remove missing values, and drop rows with negative or zero prices/quantities (these are usually returns or errors, not normal sales).

In [6]:
df_clean = df.dropna(subset=['CustomerID']).copy()
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
df_clean = df_clean.sample(5000, random_state=42)  
df_clean.shape

(5000, 8)

## 6. Engineer Features

Create a new feature, `TotalPrice` (Quantity x UnitPrice), which often carries more signal than the raw columns alone.

In [7]:
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']
df_clean[['Quantity', 'UnitPrice', 'TotalPrice']].head()

,Quantity,UnitPrice,TotalPrice
459141,6,2.08,12.48
262111,12,2.95,35.40
374705,16,0.83,13.28
263904,2,8.50,17.00
138970,200,1.65,330.00


## 7. Select Features

Choose which columns to feed into the model. We'll use the numeric features and define our target.

In [8]:
features = ['Quantity', 'UnitPrice', 'TotalPrice']
X = df_clean[features]
y = (df_clean['Country'] == 'United Kingdom').astype(int)

## 8. Split Dataset

Split into training and test sets, so we can evaluate on data the model hasn't seen.

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape

((4000, 3), (1000, 3))

## 9. Build Baseline Model

Before trying complex models, set a simple baseline: always predict the most common class. Any real model should beat this.

In [11]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score
baseline = DummyClassifier(strategy='most_frequent')
baseline.fit(X_train, y_train)
print("Baseline accuracy:", accuracy_score(y_test, baseline.predict(X_test)))

Baseline accuracy: 0.891


## 10. Train Multiple Models

Train 3 different algorithms so we can compare them fairly:
- **Logistic Regression** (simple, linear)
- **Decision Tree** (non-linear, easy to interpret)
- **Random Forest** (ensemble of trees, usually stronger)

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
models = {
    'Logistic Regression': LogisticRegression(),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42)
}
for name, model in models.items():
    model.fit(X_train, y_train)
print("All models trained.")

All models trained.


## 11. Evaluate Models

Check accuracy of each trained model on the test set.

In [13]:
for name, model in models.items():
    preds = model.predict(X_test)
    print(f"{name}: {accuracy_score(y_test, preds):.4f}")

Logistic Regression: 0.8910
Decision Tree: 0.8770
Random Forest: 0.8850


## 12. Compare Models

Put results side-by-side in a simple table for easy comparison.

In [15]:
results = pd.DataFrame({
    'Model': list(models.keys()),
    'Test Accuracy': [accuracy_score(y_test, m.predict(X_test)) for m in models.values()]
}).sort_values('Test Accuracy', ascending=False)
results

,Model,Test Accuracy
0,Logistic Regression,0.891
2,Random Forest,0.885
1,Decision Tree,0.877


## 13. Perform Cross Validation

A single test score can be misleading. Use 5-fold cross validation on each model to get a more reliable estimate of performance.

In [16]:
from sklearn.model_selection import cross_val_score
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5)
    print(f"{name}: mean={scores.mean():.4f}, std={scores.std():.4f}")

Logistic Regression: mean=0.8896, std=0.0012
Decision Tree: mean=0.8756, std=0.0098
Random Forest: mean=0.8848, std=0.0062


## 14. Tune Important Parameters

Pick the most promising model (Random Forest) and search a few key hyperparameters (`n_estimators`, `max_depth`) to improve its performance.

In [17]:
from sklearn.model_selection import GridSearchCV
param_grid = {'n_estimators': [50, 100], 'max_depth': [None, 5, 10]}
grid = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3)
grid.fit(X_train, y_train)
print("Best params:", grid.best_params_)
print("Best CV score:", grid.best_score_)

Best params: {'max_depth': 5, 'n_estimators': 100}
Best CV score: 0.8912503613159662


## 15. Select Final Model

Based on cross-validation scores and tuning results, choose the model with the best, most stable performance as our final model.

In [18]:
final_model = grid.best_estimator_
print("Final model selected:", final_model)

Final model selected: RandomForestClassifier(max_depth=5, random_state=42)


## 16. Evaluate Final Model

Check the final model's performance on the untouched test set using accuracy plus a fuller report (precision, recall, F1).

In [19]:
from sklearn.metrics import classification_report
final_preds = final_model.predict(X_test)
print("Final test accuracy:", accuracy_score(y_test, final_preds))
print(classification_report(y_test, final_preds))

Final test accuracy: 0.895
              precision    recall  f1-score   support

           0       1.00      0.04      0.07       109
           1       0.89      1.00      0.94       891

    accuracy                           0.90      1000
   macro avg       0.95      0.52      0.51      1000
weighted avg       0.91      0.90      0.85      1000



## 17. Interpret Results

Check which features matter most to the model. This helps explain *why* it makes its predictions, not just *what* it predicts.

In [20]:
importances = pd.Series(final_model.feature_importances_, index=features)
importances.sort_values(ascending=False)

TotalPrice    0.433792
Quantity      0.316110
UnitPrice     0.250098
dtype: float64

## 18. Document Limitations

- Only 2 raw features were used (`Quantity`, `UnitPrice`) — no customer history, product category, or seasonality.
- Classes are imbalanced (most orders are UK), which can inflate accuracy; precision/recall for the minority class matter more here.
- We trained on a 5,000-row sample for speed, not the full ~540,000-row dataset.
- Model was tuned on a small parameter grid; a wider search might do better.

In [21]:
print("Limitations noted above — no code needed for this step.")

Limitations noted above — no code needed for this step.


## 19. Provide Recommendations

- Collect richer features (e.g., product category, order time, customer purchase history) to improve accuracy.
- Retrain on the full dataset once the pipeline is validated on the sample.
- Use precision/recall (not just accuracy) to judge performance, given class imbalance.
- Periodically retrain the model as new order data comes in, since customer/country patterns can shift over time.

In [22]:
print("Recommendations noted above — no code needed for this step.")

Recommendations noted above — no code needed for this step.
